# Bước 4 – Xử lý Ngoại lai (Outliers Handling)

**Notebook:** `04_outliers_handling.ipynb`
**Input:** `../data/processed/step3_cleaned.csv` (output của Bước 3 – Data Cleaning)
**Output:** `../data/processed/step4_nooutliers.csv`

## Mục tiêu

Bước này chỉ tập trung vào việc **phát hiện và xử lý ngoại lai (outliers)** cho hai biến số
quan trọng nhất trong bối cảnh thương mại điện tử: `price` (giá sản phẩm) và
`freight_value` (phí vận chuyển).

Các công việc chính:
1. Kiểm tra dữ liệu đầu vào (đã qua xử lý missing/duplicates ở Bước 3).
2. Áp dụng phương pháp **IQR (Interquartile Range)** để xác định ngưỡng trên/dưới.
3. Đếm số lượng và tỷ lệ outlier cho từng biến.
4. Trực quan hóa bằng boxplot trước khi xử lý.
5. **Đánh giá thực tế** — không mặc định giá trị cao là sai — rồi mới quyết định
   loại bỏ hay giữ lại outlier, kèm lý do rõ ràng.
6. Xuất dataset kết quả cho Bước 5 (Data Encoding).

> **Phạm vi:** Notebook này **không** xử lý missing values, duplicates, encoding,
> datetime, RFM, clustering, classification hay association rules — những phần đó
> thuộc trách nhiệm của các bước/thành viên khác trong nhóm.


## Phần 2 – Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình hiển thị cho dễ đọc
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')


## Phần 3 – Đọc dữ liệu

Đọc dữ liệu đầu ra của Bước 3 (`step3_cleaned.csv`). Sử dụng đường dẫn tương đối
vì notebook nằm trong thư mục `notebooks/`.


In [ ]:
# Đường dẫn tương đối tới file input (output của Bước 3)
input_path = '../data/processed/step3_cleaned.csv'

df = pd.read_csv(input_path)
print(f"Đã đọc dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột")


## Phần 4 – Kiểm tra dữ liệu đầu vào

Trước khi phân tích outlier, cần xác nhận dữ liệu đầu vào hợp lệ: đúng shape,
kiểu dữ liệu, không còn missing values ở các cột quan trọng (vì Bước 3 đã xử lý),
và có cái nhìn tổng quan về `price`, `freight_value`.


In [ ]:
# Kích thước dữ liệu
print("Shape:", df.shape)


In [ ]:
# Thông tin kiểu dữ liệu, số lượng non-null từng cột
df.info()


In [ ]:
# Thống kê mô tả tổng quan cho toàn bộ cột số
df.describe()


In [ ]:
# Kiểm tra missing values còn sót lại (kỳ vọng = 0 vì Bước 3 đã xử lý,
# nhưng vẫn kiểm tra lại cho chắc chắn, KHÔNG tự xử lý ở đây)
missing_check = df.isna().sum()
missing_check = missing_check[missing_check > 0]
if missing_check.empty:
    print("Không còn missing values trong dữ liệu.")
else:
    print("Vẫn còn missing values (thuộc phạm vi Bước 3, chỉ ghi nhận):")
    print(missing_check)


In [ ]:
# Kiểm tra riêng hai biến sẽ phân tích outlier
df[['price', 'freight_value']].describe()


## Phần 5 – Liệt kê các biến số cần khảo sát

Theo yêu cầu của project, ta không áp dụng IQR máy móc cho mọi cột số ngay từ
đầu. Bước đầu tiên là liệt kê và xem tổng quan thống kê của toàn bộ các cột kiểu
số hiện có trong dữ liệu, làm cơ sở để so sánh và chọn lọc biến phù hợp ở
Phần 8, sau khi đã khảo sát outlier cho từng cột ở Phần 7.

In [ ]:
# Lấy toàn bộ các cột kiểu số (int64, float64) để khảo sát
all_numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
df[all_numeric_cols].describe()

## Phần 6 – Hàm tính IQR (tái sử dụng)

Xây dựng một hàm dùng chung để tính Q1, Q3, IQR và ngưỡng trên/dưới cho bất kỳ
cột số nào, tránh lặp code khi phân tích `price` và `freight_value`.


In [ ]:
def iqr_bounds(series):
    """
    Tính các thông số IQR cho một Series số.

    Công thức:
        Q1 = percentile 25%
        Q3 = percentile 75%
        IQR = Q3 - Q1
        Lower Bound = Q1 - 1.5 * IQR
        Upper Bound = Q3 + 1.5 * IQR

    Trả về: (Q1, Q3, IQR, lower_bound, upper_bound)
    """
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return Q1, Q3, IQR, lower_bound, upper_bound

## Phần 7 – Hàm tính toán outliers và khảo sát toàn bộ biến số

Dựa trên `iqr_bounds()` ở Phần 6, xây dựng hàm `outlier_report()` để tính toán
và tổng hợp outlier (số lượng, tỷ lệ phần trăm) cho nhiều cột cùng lúc. Áp dụng
hàm này cho toàn bộ các cột số đã liệt kê ở Phần 5 để có cơ sở khảo sát trước
khi chọn ra biến chính cần xử lý.

In [ ]:
def outlier_report(df, columns):
    rows = []
    for col in columns:
        Q1, Q3, IQR, lower, upper = iqr_bounds(df[col])
        mask_outlier = (df[col] < lower) | (df[col] > upper)
        n_outliers = mask_outlier.sum()

        rows.append({
            'Biến': col,
            'Mean': round(df[col].mean(), 2),
            'Median': round(df[col].median(), 2),
            'Q1': round(Q1, 2),
            'Q3': round(Q3, 2),
            'IQR': round(IQR, 2),
            'Lower Bound': round(lower, 2),
            'Upper Bound': round(upper, 2),
            'Số outlier': int(n_outliers),
            'Tỷ lệ (%)': round(n_outliers / len(df) * 100, 2)
        })

    return pd.DataFrame(rows)

In [ ]:
# Áp dụng outlier_report() cho toàn bộ các cột số để khảo sát tổng quan
outlier_report(df, all_numeric_cols)

## Phần 8 – Chọn biến chính để xử lý outlier

Từ bảng khảo sát ở Phần 7, không áp dụng IQR máy móc cho mọi cột số mà chỉ giữ
lại các biến thực sự phù hợp để phân tích ngoại lai kinh doanh:

| Biến | Quyết định | Lý do |
|---|---|---|
| `price` | Giữ | Biến kinh doanh cốt lõi, ảnh hưởng trực tiếp đến doanh thu |
| `freight_value` | Giữ | Biến kinh doanh cốt lõi, ảnh hưởng đến chi phí vận chuyển |
| `order_item_id` | Loại | Là số thứ tự sản phẩm trong đơn (1, 2, 3...), Q1 và Q3 đều bằng 1 nên IQR bằng 0, khiến mọi giá trị khác 1 bị coi là ngoại lai một cách vô nghĩa |
| `customer_zip_code_prefix` | Loại | Là mã bưu điện, một dạng định danh địa lý chứ không phải đại lượng liên tục, nên khái niệm ngoại lai không có ý nghĩa thống kê |
| `product_name_lenght`, `product_description_lenght`, `product_photos_qty`, `product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm` | Loại | Là thuộc tính mô tả và vật lý của sản phẩm, nằm ngoài phạm vi phân tích ngoại lai kinh doanh của Bước 4 |

Hai biến chính được chọn để xử lý outlier trong các phần tiếp theo là `price` và
`freight_value`.

In [ ]:
numeric_cols = ['price', 'freight_value']

# Bảng outlier chỉ cho hai biến chính đã chọn (trích từ bảng khảo sát ở Phần 7)
outlier_report(df, numeric_cols)


In [ ]:
# Lưu lại lower/upper bound (giá trị gốc, chưa làm tròn) của hai biến chính
# để dùng ở Phần 12 (Xử lý outlier)
bounds = {
    col: dict(zip(['lower', 'upper'], iqr_bounds(df[col])[3:]))
    for col in numeric_cols
}
bounds


In [ ]:
# Xem thử một vài dòng có price cao nhất để đánh giá thực tế (không kết luận vội)
top_price = df.nlargest(10, 'price')[['price', 'freight_value']]
top_price = top_price.reset_index().rename(columns={'index': 'row_index'})
top_price

In [ ]:
# Xem các dòng có freight_value cao nhất để đánh giá thực tế
top_freight = df.nlargest(10, 'freight_value')[['price', 'freight_value']]
top_freight = top_freight.reset_index().rename(columns={'index': 'row_index'})
top_freight

## Phần 9 – Boxplot trước xử lý

Trực quan hóa phân bố của `price` và `freight_value` để thấy rõ mức độ phân tán
và các điểm nằm ngoài khoảng IQR.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x=df['price'], ax=axes[0], color='#4C72B0')
axes[0].set_title('Boxplot price (trước xử lý ngoại lai)')
axes[0].set_xlabel('Price')

sns.boxplot(x=df['freight_value'], ax=axes[1], color='#DD8452')
axes[1].set_title('Boxplot freight_value (trước xử lý ngoại lai)')
axes[1].set_xlabel('Freight Value')

plt.tight_layout()
plt.show()


## Phần 10 – Đánh giá và quyết định xử lý

**Kết quả IQR (112,632 dòng):**

| Biến | Mean | Median | Q1 | Q3 | IQR | Lower Bound | Upper Bound | Outlier | Tỷ lệ |
|---|---|---|---|---|---|---|---|---|---|
| `price` | 120.65 | 74.99 | 39.90 | 134.90 | 95.00 | — (âm, không dùng) | 277.40 | 8,426 | 7.48% |
| `freight_value` | 19.99 | 16.26 | 13.08 | 21.15 | 8.07 | 0.98 | 33.25 | 12,134 | 10.77% |

**Đánh giá:**

- Tỷ lệ outlier khá cao (7.48% và 10.77%) do dữ liệu lệch phải (mean > median) — bình thường với dữ liệu giá cả, không phải lỗi.
- Top 10 giá trị cao nhất đều hợp lý (không âm, không lỗi định dạng) → nhiều khả năng là hàng cao cấp/đơn giao xa thật, không phải lỗi nhập liệu.
- Loại bỏ sẽ mất nhóm khách/sản phẩm giá trị cao, ảnh hưởng Monetary ở RFM (Bước 6).

**Quyết định:** Loại bỏ theo đúng chuẩn IQR (1.5×IQR) như yêu cầu đề bài. Outlier ở đây là dữ liệu hợp lệ, không phải lỗi — nếu cần phân tích riêng nhóm VIP sau này, nên giữ thêm `step3_cleaned.csv` để đối chiếu.

## Phần 11 – Xử lý outlier

> Code dưới đây thực hiện phương án **loại bỏ (drop)** các dòng có `price` hoặc
> `freight_value` nằm ngoài khoảng IQR — đây là phương án mặc định theo yêu cầu
> project. **Chỉ chạy/giữ lại cell này nếu quyết định ở Phần 10 là loại bỏ.**
> Nếu quyết định khác (ví dụ giữ nguyên hoặc capping), hãy chỉnh sửa logic tương ứng
> và cập nhật lại Phần 10 cho khớp.

DataFrame gốc `df` được giữ nguyên; kết quả xử lý được lưu vào `df_cleaned` mới.


In [ ]:
# Điều kiện: loại bỏ các dòng có price HOẶC freight_value là ngoại lai
# Giữ lại chỉ khi cả hai biến đều nằm trong khoảng hợp lệ
condition = (
    ((df['price'] >= bounds['price']['lower']) & (df['price'] <= bounds['price']['upper'])) &
    ((df['freight_value'] >= bounds['freight_value']['lower']) & (df['freight_value'] <= bounds['freight_value']['upper']))
)

df_cleaned = df[condition].copy()

print("Kích thước trước xử lý :", df.shape)
print("Kích thước sau xử lý   :", df_cleaned.shape)


## Phần 12 – Kiểm tra sau xử lý

In [ ]:
rows_before, cols_before = df.shape
rows_after, cols_after = df_cleaned.shape
rows_removed = rows_before - rows_after
rows_removed_ratio = rows_removed / rows_before * 100

remaining_price_outliers = (
    (df_cleaned['price'] < bounds['price']['lower']) | (df_cleaned['price'] > bounds['price']['upper'])
).sum()
remaining_freight_outliers = (
    (df_cleaned['freight_value'] < bounds['freight_value']['lower']) | (df_cleaned['freight_value'] > bounds['freight_value']['upper'])
).sum()

print(f"Số dòng trước xử lý       : {rows_before}")
print(f"Số dòng sau xử lý         : {rows_after}")
print(f"Số cột trước xử lý        : {cols_before}")
print(f"Số cột sau xử lý          : {cols_after}")
print(f"Số dòng bị loại           : {rows_removed}")
print(f"Tỷ lệ dòng bị loại        : {rows_removed_ratio:.2f}%")
print(f"Outlier price còn lại     : {remaining_price_outliers}")
print(f"Outlier freight_value còn lại : {remaining_freight_outliers}")


In [ ]:
# Boxplot sau xử lý để đối chiếu trực quan với Phần 10
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(x=df_cleaned['price'], ax=axes[0], color='#55A868')
axes[0].set_title('Boxplot price (sau xử lý ngoại lai)')
axes[0].set_xlabel('Price')

sns.boxplot(x=df_cleaned['freight_value'], ax=axes[1], color='#55A868')
axes[1].set_title('Boxplot freight_value (sau xử lý ngoại lai)')
axes[1].set_xlabel('Freight Value')

plt.tight_layout()
plt.show()


## Phần 13 – Nhận xét

- **Kết quả**: Dữ liệu giảm từ 112,632 xuống 95,075 dòng (loại 17,557 dòng, 15.59%), số cột không đổi. Outlier còn lại = 0 cho cả 2 biến, xác nhận lọc đúng.
- **Mức độ thay đổi**: 15.59% là tỷ lệ loại bỏ đáng kể (gần 1/6 dữ liệu), cần lưu ý khi báo cáo.
- **Tác động**: Dữ liệu sau xử lý phản ánh tốt nhóm khách hàng/đơn hàng phổ thông, nhưng mất phần lớn đơn hàng giá trị cao (hàng cao cấp, giao xa) — cần nêu rõ khi dùng cho RFM (Bước 6) và Clustering/Classification sau này, vì Monetary có thể bị đánh giá thấp hơn thực tế.

## Phần 14 – Xuất file

Lưu `df_cleaned` thành file output cho Bước 5 (Data Encoding).


In [ ]:
output_path = '../data/processed/step4_nooutliers.csv'

df_cleaned.to_csv(output_path, index=False)

print(f"Đã lưu dữ liệu sau xử lý ngoại lai vào: {output_path}")
print(f"Kích thước file output: {df_cleaned.shape}")
